# Song Lyrics Generator Using Subword Tokenization and LSTM in PyTorch

- Author: Jovan Heng Ghim Hong 

Goal: Build a model on pop song lyrics to generate lyrics in this style. Explore the use of subword tokenization and LSTM for text generation.

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
#imports
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

# import hugging face tokenizer
from transformers import BertTokenizer

# torch
import torch
import torch.nn as nn

# setting seeds
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

from sklearn.model_selection import train_test_split

from src.model import Model
from src.utils import timer

In [36]:
# data processing has been done inside etl.py
df = pd.read_parquet("data/pop_songs.parquet")

,tag,title,lyrics
0,pop,Wordy Rappinghood,<CHORUS> <NEWLINE> What are words worth? <NE...
1,pop,Horchata,"<VERSE> <NEWLINE> In December, drinking horc..."
2,pop,Heartless,"<CHORUS> <NEWLINE> In the night, I hear 'em ..."
3,pop,Flashing Lights,"<INTRO> <NEWLINE> Flashing lights (Lights, l..."
4,pop,Baby,"<NEWLINE> <INTRO> <NEWLINE> Oh, woah <NEWLI..."


In [ ]:
display(df.head())

In [37]:
# train test validation split 60-20-20
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
train_df, val_df = train_test_split(train_df, test_size=0.25, random_state=42)

In [ ]:
# create tokenizers for training, validation and testing
tokenizer_train = BertTokenizer.from_pretrained("bert-base-uncased")
tokenizer_val = BertTokenizer.from_pretrained("bert-base-uncased")
tokenizer_test = BertTokenizer.from_pretrained("bert-base-uncased")

# add the <VERSE> tokens to the tokenizers
sections = ['VERSE', 'CHORUS', 'BRIDGE', 'INTRO', 'OUTRO', 'PRE-CHORUS', 'HOOK']
tokenizer_train.add_tokens([f"<{section}>" for section in sections])
tokenizer_val.add_tokens([f"<{section}>" for section in sections])
tokenizer_test.add_tokens([f"<{section}>" for section in sections])

# tokenizer the lyrics
train_X = tokenizer_train(train_df["lyrics"].tolist(), padding=True, truncation=True, return_tensors="pt")["input_ids"]
val_X = tokenizer_val(val_df["lyrics"].tolist(), padding=True, truncation=True, return_tensors="pt")["input_ids"]
test_X = tokenizer_test(test_df["lyrics"].tolist(), padding=True, truncation=True, return_tensors="pt")["input_ids"]

# create the Y labels by shifting
train_Y = torch.zeros_like(train_X)
train_Y[:, :-1] = train_X[:, 1:]
train_Y[:, -1] = tokenizer_train.pad_token_id

val_Y = torch.zeros_like(val_X)
val_Y[:, :-1] = val_X[:, 1:]
val_Y[:, -1] = tokenizer_val.pad_token_id

test_Y = torch.zeros_like(test_X)
test_Y[:, :-1] = test_X[:, 1:]
test_Y[:, -1] = tokenizer_test.pad_token_id

In [ ]:
# let us create a basic lstm model
class LSTMModel(Model):
  def __init__(self, input_dim, hidden_dim, output_dim, dropout_rate=0.5):
    super().__init__()
    self.embedding = nn.Embedding(num_embeddings=30522, embedding_dim=input_dim, padding_idx=0)
    self.fc1 = nn.LSTM(input_dim, hidden_dim, batch_first=True, bidirectional=False)
    self.dropout = nn.Dropout(dropout_rate)
    self.fc2 = nn.Linear(hidden_dim * 2, 32)
    self.relu = nn.ReLU()
    self.fc3 = nn.Linear(32 , output_dim)
  
  def forward(self, x):
    x = self.embedding(x)
    _,  (h_n , _) = self.fc1(x)

    # forward
    h_forward = h_n[-2,:,:]

    h = h_forward

    out = self.dropout(h)
    out = self.fc2(out)
    out = self.relu(out)
    out = self.fc3(out)
    return out